In [1]:
from transformers import AutoTokenizer,AutoModelForTokenClassification,TrainingArguments,Trainer,DataCollatorForTokenClassification
from datasets import load_dataset
import evaluate

In [2]:
# 加载数据集
ner_dataset = load_dataset("peoples_daily_ner",cache_dir='../ner')

Generating train split:   0%|          | 0/20865 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2319 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4637 [00:00<?, ? examples/s]

In [3]:
ner_dataset['train'][0]

{'id': '0',
 'tokens': ['海',
  '钓',
  '比',
  '赛',
  '地',
  '点',
  '在',
  '厦',
  '门',
  '与',
  '金',
  '门',
  '之',
  '间',
  '的',
  '海',
  '域',
  '。'],
 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0]}

In [4]:
ner_dataset["train"].features

{'id': Value(dtype='string', id=None),
 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'], id=None), length=-1, id=None)}

In [5]:
# 存储标签映射
tag_list = ner_dataset["train"].features["ner_tags"].feature.names
tag_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

In [6]:
# 数据集预处理
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

In [7]:
tokenizer(ner_dataset["train"][0]["tokens"]) #会把每个字都单独拆出来并且加上 【cls】 和 【sep】 标签，我们需要的是【cls tokens sep】这样放在一起的整体

{'input_ids': [[101, 3862, 102], [101, 7157, 102], [101, 3683, 102], [101, 6612, 102], [101, 1765, 102], [101, 4157, 102], [101, 1762, 102], [101, 1336, 102], [101, 7305, 102], [101, 680, 102], [101, 7032, 102], [101, 7305, 102], [101, 722, 102], [101, 7313, 102], [101, 4638, 102], [101, 3862, 102], [101, 1818, 102], [101, 511, 102]], 'token_type_ids': [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]], 'attention_mask': [[1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1]]}

In [8]:
tokenizer(ner_dataset["train"][0]["tokens"],is_split_into_words=True) 

{'input_ids': [101, 3862, 7157, 3683, 6612, 1765, 4157, 1762, 1336, 7305, 680, 7032, 7305, 722, 7313, 4638, 3862, 1818, 511, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [9]:
res = tokenizer("interesting word")
res.word_ids() #word_ids 帮助定位 切分出来的ids 属于第几个词

[None, 0, 0, 0, 0, 1, None]

In [10]:
# 数据集处理
def process_function(examples):
    # 1. 对输入文本进行tokenize，注意is_split_into_words=True表示输入已经是分词后的列表
    tokenized_examples = tokenizer(examples['tokens'], max_length=128, truncation=True, is_split_into_words=True)
    
    # 2. 初始化标签列表，用于存储最终的token级别标签
    labels = []
    
    # 3. 遍历当前批次中的每个样本
    for i, label in enumerate(examples['ner_tags']):
        # 4. 获取当前样本的word_id映射关系（token到word的映射）
        word_ids = tokenized_examples.word_ids(batch_index=i)
        
        # 5. 为当前样本初始化token级别的标签列表
        label_ids = []
        
        # 6. 遍历当前样本的每个token对应的word_id
        for word_id in word_ids:
            # 7. 如果word_id为None（表示是特殊token，如[CLS], [SEP], [PAD]）
            if word_id is None:
                # 8. 特殊token的标签设为-100，这是PyTorch中CrossEntropyLoss会忽略的标签值
                label_ids.append(-100)
            else:
                # 9. 普通token的标签直接从原始标签中按word_id索引获取
                label_ids.append(label[word_id])
        
        # 10. 将当前样本处理好的token级别标签添加到labels列表中
        labels.append(label_ids)
    
    # 11. 将处理好的标签添加到tokenized_examples中
    tokenized_examples['labels'] = labels
    
    # 12. 返回处理好的样本
    return tokenized_examples

In [11]:
tokenized_dataset = ner_dataset.map(process_function,batched=True)
tokenized_dataset

Map:   0%|          | 0/20865 [00:00<?, ? examples/s]

Map:   0%|          | 0/2319 [00:00<?, ? examples/s]

Map:   0%|          | 0/4637 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20865
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2319
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4637
    })
})

In [12]:
print(tokenized_dataset["train"][0])

{'id': '0', 'tokens': ['海', '钓', '比', '赛', '地', '点', '在', '厦', '门', '与', '金', '门', '之', '间', '的', '海', '域', '。'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0], 'input_ids': [101, 3862, 7157, 3683, 6612, 1765, 4157, 1762, 1336, 7305, 680, 7032, 7305, 722, 7313, 4638, 3862, 1818, 511, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0, -100]}


In [13]:
#创建模型
model = AutoModelForTokenClassification.from_pretrained("hfl/chinese-macbert-base",num_labels = len(tag_list))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
model.config.num_labels

7

In [15]:
seqeval = evaluate.load("seqeval")
seqeval

EvaluationModule(name: "seqeval", module_type: "metric", features: {'predictions': Sequence(feature=Value(dtype='string', id='label'), length=-1, id='sequence'), 'references': Sequence(feature=Value(dtype='string', id='label'), length=-1, id='sequence')}, usage: """
Produces labelling scores along with its sufficient statistics
from a source against one or more references.

Args:
    predictions: List of List of predicted labels (Estimated targets as returned by a tagger)
    references: List of List of reference labels (Ground truth (correct) target values)
    suffix: True if the IOB prefix is after type, False otherwise. default: False
    scheme: Specify target tagging scheme. Should be one of ["IOB1", "IOB2", "IOE1", "IOE2", "IOBES", "BILOU"].
        default: None
    mode: Whether to count correct entity labels with incorrect I/B tags as true positives or not.
        If you want to only count exact matches, pass mode="strict". default: None.
    sample_weight: Array-like of sha

In [24]:
#需要将label格式转成上面的映射
import numpy as np

def eval_metric(pred):
    predictions,labels = pred
    predictions = np.argmax(predictions,axis=-1)
    
    #将id转换成字符串
    true_predictions = [
        [tag_list[p] for p,l in zip(prediction,label) if l !=-100]
        for prediction,label in zip(predictions,labels)
    ]
    
    
    true_labels = [
        [tag_list[l] for p,l in zip(prediction,label) if l !=-100]
        for prediction,label in zip(predictions,labels)
    ]
    
    result = seqeval.compute(predictions = true_predictions,references = true_labels,mode = "strict",scheme ="IOB2")
    
    return {
        "f1":result["overall_f1"],
        "accuracy":result["overall_accuracy"]
    }
    

In [25]:
#配置训练参数
args = TrainingArguments(
    output_dir="models_for_ner",
    per_device_eval_batch_size=32,
    per_gpu_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    logging_steps=50,
    num_train_epochs=1
)

In [26]:
#创建训练器
trainer = Trainer(
    model = model,
    args = args,
    tokenizer = tokenizer,
    train_dataset=tokenized_dataset["train"].select(range(100)),
    eval_dataset=tokenized_dataset["validation"].select(range(100)),
    compute_metrics=eval_metric,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)
)

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_48478/1957621097.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,No log,0.055988,0.872449,0.981547


TrainOutput(global_step=13, training_loss=0.03797969909814688, metrics={'train_runtime': 11.1061, 'train_samples_per_second': 9.004, 'train_steps_per_second': 1.171, 'total_flos': 4699471287984.0, 'train_loss': 0.03797969909814688, 'epoch': 1.0})

In [28]:
trainer.evaluate(eval_dataset=tokenized_dataset["test"].select(range(100)))

Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.


Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.


{'eval_loss': 0.06146042048931122,
 'eval_f1': 0.6176470588235293,
 'eval_accuracy': 0.9822527212494084,
 'eval_runtime': 2.4648,
 'eval_samples_per_second': 40.571,
 'eval_steps_per_second': 1.623,
 'epoch': 1.0}

In [29]:
#模型预测
from transformers import pipeline

# 使用pipeline 进行推理 需要置顶id2label
model.config.id2label = {idx:label for idx,label in enumerate(tag_list)}
model.config


BertConfig {
  "_name_or_path": "hfl/chinese-macbert-base",
  "architectures": [
    "BertForTokenClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-PER",
    "2": "I-PER",
    "3": "B-ORG",
    "4": "I-ORG",
    "5": "B-LOC",
    "6": "I-LOC"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  

In [30]:
ner_pipe = pipeline("token-classification",model = model,tokenizer = tokenizer,device = 0,aggregation_strategy="simple")
ner_pipe

Device set to use mps:0


In [31]:
res = ner_pipe('长城是北京的吗')
res 

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'LOC',
  'score': np.float32(0.9334517),
  'word': '长 城',
  'start': 0,
  'end': 2},
 {'entity_group': 'LOC',
  'score': np.float32(0.966657),
  'word': '北 京',
  'start': 3,
  'end': 5}]

In [32]:
x = "长城是北京的吗"
ner_result = {}
for r in res:
    if r["entity_group"] not in ner_result:
        ner_result[r["entity_group"]] = []
    ner_result[r["entity_group"]].append(x[r["start"]:r["end"]])

ner_result

{'LOC': ['长城', '北京']}

In [35]:
#使用baseline模型直接进行推理
#即使训练100条 效果也是好很多的
model_base = AutoModelForTokenClassification.from_pretrained("hfl/chinese-macbert-base",num_labels = len(tag_list))
model_base.config.id2label = {idx:label for idx,label in enumerate(tag_list)}

ner_pipe_base = pipeline("token-classification",model = model_base,tokenizer = tokenizer,device = 0,aggregation_strategy="simple")
res2 =ner_pipe_base('长城是北京的吗')
res2

Some weights of BertForTokenClassification were not initialized from the model checkpoint at hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


[{'entity_group': 'PER',
  'score': np.float32(0.19006447),
  'word': '城',
  'start': 1,
  'end': 2},
 {'entity_group': 'PER',
  'score': np.float32(0.23158436),
  'word': '是',
  'start': 2,
  'end': 3},
 {'entity_group': 'PER',
  'score': np.float32(0.2667485),
  'word': '北',
  'start': 3,
  'end': 4},
 {'entity_group': 'PER',
  'score': np.float32(0.23062429),
  'word': '京',
  'start': 4,
  'end': 5},
 {'entity_group': 'PER',
  'score': np.float32(0.22622101),
  'word': '的',
  'start': 5,
  'end': 6},
 {'entity_group': 'PER',
  'score': np.float32(0.31499526),
  'word': '吗',
  'start': 6,
  'end': 7}]